In [ ]:
!pip install pandas numpy scikit-learn

import pandas as pd
import numpy as np
from google.colab import files

from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.model_selection import KFold
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_absolute_error, recall_score

# Upload dataset
uploaded = files.upload()

df = pd.read_csv("ML_Final_Water_Disease_Training_Dataset_with_sources.csv")

print("Dataset shape:", df.shape)

# Encode categorical column
encoder = LabelEncoder()
df["Water Source Type"] = encoder.fit_transform(df["Water Source Type"])

# Input features
X = df[
    [
        "Water Source Type",
        "Contaminant Level (ppm)",
        "pH Level",
        "Turbidity (NTU)",
        "Dissolved Oxygen (mg/L)",
        "Nitrate Level (mg/L)",
        "Bacteria Count (CFU/mL)",
        "Fecal Coliform",
        "Total Coliform",
        "Temperature"
    ]
]

# Targets
y_d = df["Diarrheal Cases Likelihood (%)"]
y_c = df["Cholera Cases Likelihood (%)"]
y_t = df["Typhoid Cases Likelihood (%)"]

kf = KFold(n_splits=5, shuffle=True, random_state=42)

r2_scores=[]
mae_scores=[]
recall_scores=[]

for train_idx, test_idx in kf.split(X):

    X_train,X_test=X.iloc[train_idx],X.iloc[test_idx]

    y_d_train,y_d_test=y_d.iloc[train_idx],y_d.iloc[test_idx]
    y_c_train,y_c_test=y_c.iloc[train_idx],y_c.iloc[test_idx]
    y_t_train,y_t_test=y_t.iloc[train_idx],y_t.iloc[test_idx]

    scaler=RobustScaler()

    X_train=scaler.fit_transform(X_train)
    X_test=scaler.transform(X_test)

    svm_d=SVR(kernel='rbf',C=50,gamma=0.2,epsilon=0.05)
    svm_c=SVR(kernel='rbf',C=50,gamma=0.2,epsilon=0.05)
    svm_t=SVR(kernel='rbf',C=50,gamma=0.2,epsilon=0.05)

    knn_d=KNeighborsRegressor(n_neighbors=5)
    knn_c=KNeighborsRegressor(n_neighbors=5)
    knn_t=KNeighborsRegressor(n_neighbors=5)

    svm_d.fit(X_train,y_d_train)
    svm_c.fit(X_train,y_c_train)
    svm_t.fit(X_train,y_t_train)

    knn_d.fit(X_train,y_d_train)
    knn_c.fit(X_train,y_c_train)
    knn_t.fit(X_train,y_t_train)

    pred_d=0.7*svm_d.predict(X_test)+0.3*knn_d.predict(X_test)
    pred_c=0.7*svm_c.predict(X_test)+0.3*knn_c.predict(X_test)
    pred_t=0.7*svm_t.predict(X_test)+0.3*knn_t.predict(X_test)

    pred=np.vstack([pred_d,pred_c,pred_t]).T
    true=np.vstack([y_d_test,y_c_test,y_t_test]).T

    r2_scores.append(r2_score(true,pred))
    mae_scores.append(mean_absolute_error(true,pred))

    threshold=20
    recall=recall_score(
        (true>threshold).astype(int).flatten(),
        (pred>threshold).astype(int).flatten()
    )

    recall_scores.append(recall)

print("\nModel Performance")
print("Average R²:",np.mean(r2_scores))
print("Average MAE:",np.mean(mae_scores))
print("Average Recall:",np.mean(recall_scores))

# Initialize and fit a final scaler on the entire dataset X for new predictions
scaler_final = RobustScaler()
scaler_final.fit(X)

Saving ML_Final_Water_Disease_Training_Dataset_with_sources.csv to ML_Final_Water_Disease_Training_Dataset_with_sources (1).csv
Dataset shape: (1163, 13)

Model Performance
Average R²: 0.8739187255980306
Average MAE: 4.2274704813725945
Average Recall: 0.8299495751706145


RobustScaler()

In [7]:
X_new = pd.DataFrame({

"Water Source Type": ["River"],
"Contaminant Level (ppm)": [0.06],
"pH Level": [6.11],
"Turbidity (NTU)": [3.12],
"Dissolved Oxygen (mg/L)": [6.97],
"Nitrate Level (mg/L)": [44.98],
"Bacteria Count (CFU/mL)": [2977],
"Fecal Coliform": [2000],
"Total Coliform": [8000],
"Temperature": [22.42]

})

X_new["Water Source Type"] = encoder.transform(X_new["Water Source Type"])

X_new_scaled = scaler_final.transform(X_new)

# Predictions
diarrhea = 0.7 * svm_d.predict(X_new_scaled) + 0.3 * knn_d.predict(X_new_scaled)
cholera = 0.7 * svm_c.predict(X_new_scaled) + 0.3 * knn_c.predict(X_new_scaled)
typhoid = 0.7 * svm_t.predict(X_new_scaled) + 0.3 * knn_t.predict(X_new_scaled)

print("\nPredicted Disease Likelihoods (%)")
print("Diarrhea:", diarrhea[0])
print("Cholera:", cholera[0])
print("Typhoid:", typhoid[0])


Predicted Disease Likelihoods (%)
Diarrhea: 46.4553863406136
Cholera: 45.3087547345474
Typhoid: 48.2720246790153


### Predicting Likelihoods for `new_data_df`

In [6]:
# Create a copy to avoid modifying the original new_data_df directly
processed_new_data_df = new_data_df.copy()

# Encode 'Water Source Type' using the previously fitted encoder
# Handle potential new categories not seen during training
# If a new category is encountered, it will raise a ValueError, which we catch.
# In a production system, you might want to handle new categories more gracefully (e.g., assign to 'Other' or use one-hot encoding with handle_unknown='ignore').
try:
    processed_new_data_df["Water Source Type"] = encoder.transform(processed_new_data_df["Water Source Type"])
except ValueError as e:
    print(f"Warning: {e}. One or more 'Water Source Type' categories in your new data were not seen during training. This might cause issues or lead to inaccurate predictions.")
    print("Please ensure your new data contains 'Water Source Type' values that were present in the training data, or update the encoder accordingly.")
    # For demonstration, we'll try to proceed, but be aware of this potential issue.
    # You might want to implement a more robust strategy for unknown categories.
    # For now, we'll just re-raise if it's critical, or let it pass if acceptable for the context.


# Select the same input features as used for training
X_new_for_prediction = processed_new_data_df[
    [
        "Water Source Type",
        "Contaminant Level (ppm)",
        "pH Level",
        "Turbidity (NTU)",
        "Dissolved Oxygen (mg/L)",
        "Nitrate Level (mg/L)",
        "Bacteria Count (CFU/mL)",
        "Fecal Coliform",
        "Total Coliform",
        "Temperature"
    ]
]

# Scale the new input features using the final scaler fitted on the entire training data
X_new_scaled_for_prediction = scaler_final.transform(X_new_for_prediction)

# Make predictions using the trained ensemble models
pred_d_new_data = 0.7 * svm_d.predict(X_new_scaled_for_prediction) + 0.3 * knn_d.predict(X_new_scaled_for_prediction)
pred_c_new_data = 0.7 * svm_c.predict(X_new_scaled_for_prediction) + 0.3 * knn_c.predict(X_new_scaled_for_prediction)
pred_t_new_data = 0.7 * svm_t.predict(X_new_scaled_for_prediction) + 0.3 * knn_t.predict(X_new_scaled_for_prediction)

# Add predictions to the original new_data_df for easy viewing
new_data_df['Predicted Diarrheal Cases Likelihood (%)'] = pred_d_new_data
new_data_df['Predicted Cholera Cases Likelihood (%)'] = pred_c_new_data
new_data_df['Predicted Typhoid Cases Likelihood (%)'] = pred_t_new_data

print("\nPredicted Disease Likelihoods for the new data:")
display(new_data_df[['Water Source Type', 'Predicted Diarrheal Cases Likelihood (%)', 'Predicted Cholera Cases Likelihood (%)', 'Predicted Typhoid Cases Likelihood (%)']])


Predicted Disease Likelihoods for the new data:


,Water Source Type,Predicted Diarrheal Cases Likelihood (%),Predicted Cholera Cases Likelihood (%),Predicted Typhoid Cases Likelihood (%)
0,Lake,39.346231,38.403289,38.768569
1,Well,11.664900,18.572876,18.578353
2,River,47.788257,45.932226,48.383107


### Define Your New Data

Below is a sample `pd.DataFrame` with the structure expected by the model. You can modify this data directly, add more rows, or define your own entirely new data within this cell. Please ensure all required columns are present and correctly named.

In [11]:
new_data_df = pd.DataFrame({
    "Water Source Type": ["Lake", "Well", "River"],
    "Contaminant Level (ppm)": [25.0, 10.5, 35.2],
    "pH Level": [6.8, 7.5, 6.2],
    "Turbidity (NTU)": [12.0, 3.1, 20.5],
    "Dissolved Oxygen (mg/L)": [5.5, 8.0, 4.0],
    "Nitrate Level (mg/L)": [12.0, 4.5, 18.0],
    "Bacteria Count (CFU/mL)": [30000, 500, 75000],
    "Fecal Coliform": [500, 50, 2000],
    "Total Coliform": [2500, 200, 10000],
    "Temperature": [24.0, 18.5, 28.0]
})

display(new_data_df)

diarrhea = 0.7 * svm_d.predict(X_new_scaled) + 0.3 * knn_d.predict(X_new_scaled)
cholera = 0.7 * svm_c.predict(X_new_scaled) + 0.3 * knn_c.predict(X_new_scaled)
typhoid = 0.7 * svm_t.predict(X_new_scaled) + 0.3 * knn_t.predict(X_new_scaled)

print("\nPredicted Disease Likelihoods (%)")
print("Diarrhea:", diarrhea[0])
print("Cholera:", cholera[0])
print("Typhoid:", typhoid[0])

,Water Source Type,Contaminant Level (ppm),pH Level,Turbidity (NTU),Dissolved Oxygen (mg/L),Nitrate Level (mg/L),Bacteria Count (CFU/mL),Fecal Coliform,Total Coliform,Temperature
0,Lake,25.0,6.8,12.0,5.5,12.0,30000,500,2500,24.0
1,Well,10.5,7.5,3.1,8.0,4.5,500,50,200,18.5
2,River,35.2,6.2,20.5,4.0,18.0,75000,2000,10000,28.0



Predicted Disease Likelihoods (%)
Diarrhea: 46.4553863406136
Cholera: 45.3087547345474
Typhoid: 48.2720246790153


Now that you have defined `new_data_df`, please tell me what specific changes you'd like to make to it.